In [51]:
import pyautogui as pyg
pyg.sleep(2)
basex=326
basey=391
gap=40

In [52]:
for row in range(8):
    for col in range(8):
        x = basex + col * gap
        y = basey + row * gap
        color = pyg.pixel(x, y)
        print(row, col, x, y, color)


0 0 326 391 (24, 15, 19)
0 1 366 391 (24, 16, 19)
0 2 406 391 (25, 16, 19)
0 3 446 391 (25, 17, 20)
0 4 486 391 (25, 17, 20)
0 5 526 391 (26, 18, 20)
0 6 566 391 (27, 18, 20)
0 7 606 391 (30, 20, 21)
1 0 326 431 (33, 23, 23)
1 1 366 431 (32, 23, 22)
1 2 406 431 (32, 22, 22)
1 3 446 431 (32, 22, 22)
1 4 486 431 (32, 22, 22)
1 5 526 431 (33, 23, 22)
1 6 566 431 (33, 23, 23)
1 7 606 431 (35, 24, 23)
2 0 326 471 (35, 25, 24)
2 1 366 471 (36, 26, 25)
2 2 406 471 (37, 28, 27)
2 3 446 471 (37, 28, 27)
2 4 486 471 (36, 25, 23)
2 5 526 471 (36, 26, 24)
2 6 566 471 (36, 26, 25)
2 7 606 471 (36, 26, 25)
3 0 326 511 (36, 26, 25)
3 1 366 511 (37, 29, 28)
3 2 406 511 (37, 28, 28)
3 3 446 511 (37, 28, 27)
3 4 486 511 (37, 28, 27)
3 5 526 511 (37, 30, 26)
3 6 566 511 (37, 27, 28)
3 7 606 511 (37, 27, 28)
4 0 326 551 (37, 28, 26)
4 1 366 551 (37, 28, 26)
4 2 406 551 (37, 27, 28)
4 3 446 551 (37, 27, 26)
4 4 486 551 (37, 27, 27)
4 5 526 551 (37, 27, 27)
4 6 566 551 (37, 27, 27)
4 7 606 551 (37, 26, 27)


In [53]:
def get_gem_position(r, c):
    x = basex + c * gap
    y = basey + r * gap
    return (x, y)

In [54]:
def get_gem_color(r, c, im):
    x, y = get_gem_position(r, c)
    return im.getpixel((x, y))

In [55]:
def can_move_left(r, c, im):
    if not (0 <= r < 8 and 1 <= c < 8):
        return False

    left_c = c - 1
    left_color = get_gem_color(r, left_c, im)
    current_color = get_gem_color(r, c, im)

    def color_after_swap(row, col):
        if (row, col) == (r, left_c):
            return current_color
        if (row, col) == (r, c):
            return left_color
        return get_gem_color(row, col, im)

    def makes_match(row, col):
        color = color_after_swap(row, col)
        for dr, dc in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                nr = row + dr * direction
                nc = col + dc * direction
                while (0 <= nr < 8 and 0 <= nc < 8
                       and color_after_swap(nr, nc) == color):
                    count += 1
                    nr += dr * direction
                    nc += dc * direction
            if count >= 3:
                return True
        return False

    return makes_match(r, left_c) or makes_match(r, c)

In [56]:
def _can_swap_and_match(r, c, dr, dc, im):
    nr = r + dr
    nc = c + dc
    if not (0 <= r < 8 and 0 <= c < 8 and 0 <= nr < 8 and 0 <= nc < 8):
        return False

    current_color = get_gem_color(r, c, im)
    neighbor_color = get_gem_color(nr, nc, im)

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return neighbor_color
        if (row, col) == (nr, nc):
            return current_color
        return get_gem_color(row, col, im)

    def makes_match(row, col):
        color = color_after_swap(row, col)
        for axis_r, axis_c in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                check_r = row + axis_r * direction
                check_c = col + axis_c * direction
                while (0 <= check_r < 8 and 0 <= check_c < 8
                       and color_after_swap(check_r, check_c) == color):
                    count += 1
                    check_r += axis_r * direction
                    check_c += axis_c * direction
            if count >= 3:
                return True
        return False

    return makes_match(r, c) or makes_match(nr, nc)

def can_move_right(r, c, im):
    return _can_swap_and_match(r, c, 0, 1, im)

def can_move_up(r, c, im):
    return _can_swap_and_match(r, c, -1, 0, im)

def can_move_down(r, c, im):
    return _can_swap_and_match(r, c, 1, 0, im)

In [57]:
im = pyg.screenshot()
print(can_move_left(5, 4, im))

False


In [58]:
print(get_gem_position(0,0))
print(get_gem_position(0,1))
print(get_gem_position(1,0))

(326, 391)
(366, 391)
(326, 431)


In [59]:
im = pyg.screenshot()
for i in range(8):
    x, y = get_gem_position(0, i)
    pyg.click(x, y, duration=1)
    print(get_gem_color(0, i, im))

(252, 234, 107)
(237, 15, 237)
(237, 15, 237)
(70, 245, 115)
(237, 15, 237)
(253, 28, 57)
(70, 245, 115)
(18, 137, 251)


In [60]:
import time

start_time = time.time()
directions = (
    (can_move_left, 0, -1),
    (can_move_right, 0, 1),
    (can_move_up, -1, 0),
    (can_move_down, 1, 0),
)

while time.time() - start_time < 60:
    # 每輪掃描都先取得最新棋盤畫面。
    im = pyg.screenshot()
    move_found = False

    for r in range(8):
        for c in range(8):
            if time.time() - start_time >= 60:
                break

            for can_move, dr, dc in directions:
                if time.time() - start_time >= 60:
                    break
                if can_move(r, c, im):
                    x1, y1 = get_gem_position(r, c)
                    x2, y2 = get_gem_position(r + dr, c + dc)
                    pyg.moveTo(x1, y1)
                    pyg.dragTo(x2, y2, duration=0.2)
                    pyg.sleep(0.1)
                    # 交換後立即更新畫面，下一輪從棋盤起點重新掃描。
                    im = pyg.screenshot()
                    move_found = True
                    break

            if move_found or time.time() - start_time >= 60:
                break

        if move_found or time.time() - start_time >= 60:
            break

    if not move_found:
        # 無可消除組合時，本輪掃描結束後仍更新畫面再重試。
        im = pyg.screenshot()
        pyg.sleep(0.05)